# Groundedness Evaluation with Large Context

## Overview
This notebook tests the groundedness evaluation with very large context to reproduce and investigate the issue where large contexts cause 400 (Bad Request) errors.

**The Problem:**
When the context becomes very large, the groundedness evaluation API call starts returning a 400 (Bad Request) error. Our assumption is that this might be due to the prompt exceeding the model's context window.

**What this notebook does:**
1. Load a long context from file with option to artificially increase size by repetition
2. Define a test query and response based on the context
3. Track detailed token counts showing breakdown: user question, system message, context, and response
4. Run groundedness evaluation and track token usage
5. Compare token consumption and analyze limits

#### Prerequisites
- Azure OpenAI resource with model deployments
- Azure AI evaluation SDK installed
- Credentials configured in .env file

## Table of Contents

[Install Required Libraries](#install-required-libraries)  
[Load Configuration](#load-configuration)  
[Load and Prepare Context](#load-and-prepare-context)  
[Define Test Query and Response](#define-test-query-and-response)  
[Count Tokens in Components](#count-tokens-in-components)  
[Run Groundedness Evaluation](#run-groundedness-evaluation)  
[Analyze Results](#analyze-results)

## Install Required Libraries

In [ ]:
# Install required packages
%pip install azure-ai-evaluation openai python-dotenv tiktoken --quiet

## Load Configuration

Load credentials and model configuration from environment file.

In [3]:
import os
from dotenv import load_dotenv
from azure.ai.evaluation import AzureOpenAIModelConfiguration

# Load environment variables
env_loaded = load_dotenv('.env')
print(f"Environment file loaded: {env_loaded}")

# Azure OpenAI configuration
api_key = os.getenv("OPENAI_API_KEY")
endpoint = os.getenv("OPENAI_API_ENDPOINT")
deployment = os.getenv("GPT4_MODEL_NAME")  # Using GPT-4 for better quality
api_version = os.getenv("OPENAI_API_VERSION", "2024-02-15-preview")

# Display what we got (helpful for debugging)
print(f"\nLoaded configuration:")
print(f"  Endpoint: {endpoint if endpoint else '❌ NOT SET'}")
print(f"  API Key: {'✓ Set' if api_key else '❌ NOT SET'}")
print(f"  Deployment: {deployment if deployment else '❌ NOT SET'}")
print(f"  API Version: {api_version}")

# Check for issues
if not endpoint or not endpoint.startswith(('http://', 'https://')):
    print(f"\n⚠️ WARNING: Endpoint issue detected!")
    print(f"   Expected format: https://your-resource.openai.azure.com/")
    print(f"   Update your .env file with the correct endpoint")

# Model configuration for evaluation
if endpoint and api_key and deployment:
    model_config = AzureOpenAIModelConfiguration(
        azure_endpoint=endpoint,
        api_key=api_key,
        azure_deployment=deployment,
        api_version=api_version
    )
    print(f"\n✓ Configuration object created successfully")
else:
    print(f"\n❌ Cannot create configuration - missing required values")
    print(f"   Please check your .env file")

Environment file loaded: True

Loaded configuration:
  Endpoint: https://openai-universal.openai.azure.com/
  API Key: ✓ Set
  Deployment: gpt-4
  API Version: 2023-05-15

✓ Configuration object created successfully


## Load and Prepare Context

Load the long context from file and optionally repeat it to increase size.

In [ ]:
# Read the long context file
context_file = "long_context.txt"

with open(context_file, 'r', encoding='utf-8') as f:
    base_context = f.read()

# Configuration: How many times to repeat the context
# Increase this number to artificially increase context size
CONTEXT_REPETITIONS = 1  # Start with 1 (no repetition), increase to test limits

# Create the full context by repeating
if CONTEXT_REPETITIONS > 1:
    full_context = "\n\n---REPEATED SECTION---\n\n".join([base_context] * CONTEXT_REPETITIONS)
else:
    full_context = base_context

print(f"Base context length: {len(base_context)} characters")
print(f"Context repetitions: {CONTEXT_REPETITIONS}")
print(f"Full context length: {len(full_context)} characters")
print(f"\nFirst 500 characters of context:\n{full_context[:500]}...")

Base context length: 15603 characters
Context repetitions: 100
Full context length: 1562874 characters

First 500 characters of context:
Marie Curie: A Comprehensive Biography

Early Life and Education

Maria Sklodowska, later known as Marie Curie, was born on November 7, 1867, in Warsaw, Poland, which at that time was part of the Russian Empire. She was the youngest of five children born to Wladyslaw Sklodowski, a mathematics and physics teacher, and Bronislawa Boguska Sklodowska, who was a teacher, pianist, and singer. Her childhood was marked by both intellectual stimulation and tragedy. When Marie was only ten years old, her ...


## Generate Response with OpenAI

Make an actual OpenAI call with:
- System message defining the assistant's role
- User query asking a complex question
- Context provided in the user message
- Track token usage for all components

In [5]:
from openai import AzureOpenAI

# Initialize OpenAI client
client = AzureOpenAI(
    api_key=api_key,
    api_version=api_version,
    azure_endpoint=endpoint
)

# System message - defines the assistant's role
system_message = """You are a highly knowledgeable and precise research assistant with expertise in analyzing and synthesizing information from provided contexts. Your primary responsibilities include:

1. Carefully reading and understanding all provided context material
2. Extracting relevant information that directly addresses the user's query
3. Providing accurate, comprehensive, and well-structured responses
4. Always citing specific information, facts, dates, and details from the context
5. Organizing your responses in a clear, logical manner with proper flow
6. Distinguishing between explicitly stated information and reasonable inferences
7. Acknowledging when information is not available in the provided context
8. Maintaining objectivity and avoiding speculation beyond what the context supports

When responding:
- Begin with the most important information
- Use specific quotes or paraphrases from the context
- Provide complete answers that address all parts of multi-part questions
- Use clear paragraph structure for readability
- Include relevant context and background information when helpful
- Be thorough but concise, avoiding unnecessary verbosity

Your goal is to be a reliable, accurate, and helpful assistant that users can trust for well-grounded, context-based information."""

# User query - a more involved, multi-part question
user_query = """
What were Marie Curie's major scientific achievements and how did her work impact both the scientific community and society? 
Please include details about her discoveries, any awards she received, and her contributions during World War I.

Context:
{context}"""

# Create the full user message with context
user_message = user_query.format(context=full_context)

print("Making OpenAI API call...")
print(f"System message length: {len(system_message)} chars")
print(f"Context length: {len(full_context)} chars")
print(f"User query length: {len(user_query)} chars")

# Make the API call
response = client.chat.completions.create(
    model=deployment,
    messages=[
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message}
    ],
    temperature=0.3,
    max_tokens=5000
)

# Extract the response
test_response = response.choices[0].message.content
test_query = "What were Marie Curie's major scientific achievements and how did her work impact both the scientific community and society? Please include details about her discoveries, any awards she received, and her contributions during World War I."

# Track token usage from the API call
api_tokens = {
    "prompt_tokens": response.usage.prompt_tokens,
    "completion_tokens": response.usage.completion_tokens,
    "total_tokens": response.usage.total_tokens
}

print(f"\n✓ API call completed successfully")
print(f"\nQuery: {test_query}")
print(f"\nResponse preview (first 300 chars):\n{test_response[:300]}...")
print(f"\n✓ Token usage from OpenAI API:")
print(f"  Prompt tokens:      {api_tokens['prompt_tokens']:,}")
print(f"  Completion tokens:  {api_tokens['completion_tokens']:,}")
print(f"  Total tokens:       {api_tokens['total_tokens']:,}")

Making OpenAI API call...
System message length: 1286 chars
Context length: 15603 chars
User query length: 259 chars

✓ API call completed successfully

Query: What were Marie Curie's major scientific achievements and how did her work impact both the scientific community and society? Please include details about her discoveries, any awards she received, and her contributions during World War I.

Response preview (first 300 chars):
Marie Curie’s scientific achievements and their impact on both the scientific community and society are profound and multifaceted. Below is a detailed account of her contributions, awards, and her role during World War I, based on the provided context:

### Major Scientific Achievements
1. **Discove...

✓ Token usage from OpenAI API:
  Prompt tokens:      3,187
  Completion tokens:  1,245
  Total tokens:       4,432


## Count Tokens in Components

Count the tokens in each component showing detailed breakdown of:
- User question tokens
- System message tokens
- Context tokens
- Response tokens

In [6]:
import tiktoken

# Get the encoding for the model
# GPT-4 and GPT-3.5-turbo use cl100k_base encoding
encoding = tiktoken.get_encoding("cl100k_base")

# Count tokens for each component
def count_tokens(text):
    """Count tokens in a text string."""
    return len(encoding.encode(text))

# Count tokens for each component from our OpenAI call
user_question_tokens = count_tokens(test_query)
system_message_tokens = count_tokens(system_message)
context_tokens = count_tokens(full_context)
response_tokens = count_tokens(test_response)

# For groundedness evaluation, estimate additional system prompt tokens
estimated_eval_system_tokens = 500  # Conservative estimate for evaluation system prompt

# Calculate totals
# For the OpenAI generation call
generation_prompt_tokens = user_question_tokens + system_message_tokens + context_tokens

# For the groundedness evaluation call
eval_prompt_tokens = user_question_tokens + estimated_eval_system_tokens + context_tokens + response_tokens

print("=" * 80)
print("DETAILED TOKEN BREAKDOWN")
print("=" * 80)

print(f"\n1. User Question:")
print(f"   {test_query[:80]}...")
print(f"   Tokens: {user_question_tokens:,}")

print(f"\n2. System Message (our call):")
print(f"   {system_message[:80]}...")
print(f"   Tokens: {system_message_tokens:,}")

print(f"\n3. Context:")
print(f"   Characters: {len(full_context):,}")
print(f"   Tokens: {context_tokens:,}")
print(f"   Repetitions: {CONTEXT_REPETITIONS}x")

print(f"\n4. Response from OpenAI:")
print(f"   Characters: {len(test_response):,}")
print(f"   Tokens: {response_tokens:,}")
print(f"   Preview: {test_response[:80]}...")

print(f"\n" + "=" * 80)
print(f"TOKEN USAGE COMPARISON")
print(f"=" * 80)

print(f"\nOpenAI Generation Call:")
print(f"  User Question:     {user_question_tokens:>10,} tokens")
print(f"  System Message:    {system_message_tokens:>10,} tokens")
print(f"  Context:           {context_tokens:>10,} tokens")
print(f"  {'-' * 50}")
print(f"  TOTAL Prompt:      {generation_prompt_tokens:>10,} tokens")
print(f"  Response:          {response_tokens:>10,} tokens")
print(f"  {'-' * 50}")
print(f"  GRAND TOTAL:       {generation_prompt_tokens + response_tokens:>10,} tokens")
print(f"  (API reported:     {api_tokens['total_tokens']:>10,} tokens)")

print(f"\nGroundedness Evaluation (estimated):")
print(f"  User Question:     {user_question_tokens:>10,} tokens")
print(f"  Eval System Msg:   {estimated_eval_system_tokens:>10,} tokens (estimated)")
print(f"  Context:           {context_tokens:>10,} tokens")
print(f"  Response:          {response_tokens:>10,} tokens")
print(f"  {'-' * 50}")
print(f"  TOTAL Estimated:   {eval_prompt_tokens:>10,} tokens")

print(f"\n" + "=" * 80)

# Check against common model limits
model_limits = {
    "gpt-35-turbo": 4096,
    "gpt-35-turbo-16k": 16384,
    "gpt-4": 8192,
    "gpt-4-32k": 32768,
    "gpt-4-turbo": 128000,
    "gpt-4o": 128000
}

print(f"\nModel Context Window Comparison (for Evaluation):")
for model_name, limit in model_limits.items():
    percentage = (eval_prompt_tokens / limit) * 100
    if eval_prompt_tokens < limit * 0.9:
        status = "✓ OK"
    elif eval_prompt_tokens < limit:
        status = "⚠ NEAR LIMIT"
    else:
        status = "❌ EXCEEDS"
    print(f"  {model_name:20s}: {limit:>10,} tokens  [{status}] ({percentage:>6.1f}% used)")

DETAILED TOKEN BREAKDOWN

1. User Question:
   What were Marie Curie's major scientific achievements and how did her work impac...
   Tokens: 42

2. System Message (our call):
   You are a highly knowledgeable and precise research assistant with expertise in ...
   Tokens: 231

3. Context:
   Characters: 15,603
   Tokens: 2,945
   Repetitions: 1x

4. Response from OpenAI:
   Characters: 6,323
   Tokens: 1,255
   Preview: Marie Curie’s scientific achievements and their impact on both the scientific co...

TOKEN USAGE COMPARISON

OpenAI Generation Call:
  User Question:             42 tokens
  System Message:           231 tokens
  Context:                2,945 tokens
  --------------------------------------------------
  TOTAL Prompt:           3,218 tokens
  Response:               1,255 tokens
  --------------------------------------------------
  GRAND TOTAL:            4,473 tokens
  (API reported:          4,432 tokens)

Groundedness Evaluation (estimated):
  User Question:        

## Run Groundedness Evaluation

Now run the groundedness evaluation and track token usage.

In [7]:
from azure.ai.evaluation import GroundednessEvaluator

print("Initializing Groundedness Evaluator...")
print(f"Using model: {deployment}")

# Initialize evaluator with threshold
groundedness_evaluator = GroundednessEvaluator(
    model_config=model_config,
    threshold=3
)

print("\nRunning groundedness evaluation...")
print(f"  Query length: {len(test_query)} chars")
print(f"  Context length: {len(full_context)} chars")
print(f"  Response length: {len(test_response)} chars")

try:
    # Run evaluation
    groundedness_result = groundedness_evaluator(
        query=test_query,
        context=full_context,
        response=test_response
    )
    
    print("\n✓ Groundedness evaluation completed successfully!")
    print(f"\nResults:")
    print(f"  Score: {groundedness_result.get('groundedness', 'N/A')}")
    print(f"  Reasoning: {groundedness_result.get('groundedness_reason', 'N/A')}")
    
    # Try to extract token usage from result if available
    # Note: The evaluation SDK may not always expose token counts directly
    if 'token_usage' in groundedness_result:
        eval_tokens = groundedness_result['token_usage']
        print(f"\n✓ Token usage for evaluation:")
        print(f"  Prompt tokens: {eval_tokens.get('prompt_tokens', 'N/A')}")
        print(f"  Completion tokens: {eval_tokens.get('completion_tokens', 'N/A')}")
        print(f"  Total tokens: {eval_tokens.get('total_tokens', 'N/A')}")
    else:
        print(f"\n⚠ Token usage not directly available in evaluation result")
        print(f"  (Estimated from component sizes: ~{eval_prompt_tokens:,} tokens)")
    
except Exception as e:
    print(f"\n❌ Groundedness evaluation failed!")
    print(f"Error type: {type(e).__name__}")
    print(f"Error message: {str(e)}")
    
    # Check if it's the expected 400 error
    if "400" in str(e) or "Bad Request" in str(e):
        print(f"\n⚠ This is the expected 400 (Bad Request) error!")
        print(f"  Likely cause: Total tokens ({eval_prompt_tokens:,}) exceeded model limits")
        print(f"  Context alone: {context_tokens:,} tokens")
        print(f"\nSuggestions:")
        print(f"  1. Reduce CONTEXT_REPETITIONS value")
        print(f"  2. Use a model with larger context window (e.g., GPT-4-turbo or GPT-4o)")
        print(f"  3. Implement chunking strategy for large contexts")
    
    groundedness_result = None

Initializing Groundedness Evaluator...
Using model: gpt-4

Running groundedness evaluation...
  Query length: 237 chars
  Context length: 15603 chars
  Response length: 6323 chars

✓ Groundedness evaluation completed successfully!

Results:
  Score: 5.0
  Reasoning: The RESPONSE is fully grounded in the CONTEXT, directly answers the QUERY, and includes all relevant details about Marie Curie's achievements, awards, and contributions during World War I. It is accurate, complete, and well-supported by the CONTEXT.

⚠ Token usage not directly available in evaluation result
  (Estimated from component sizes: ~4,742 tokens)


## Analyze Results

Compare the token usage and analyze the results.

In [ ]:
print("=" * 80)
print("SUMMARY: TOKEN USAGE ANALYSIS")
print("=" * 80)

print(f"\n1. OpenAI Generation Call:")
print(f"   Prompt tokens:      {api_tokens['prompt_tokens']:>10,}")
print(f"   Completion tokens:  {api_tokens['completion_tokens']:>10,}")
print(f"   Total tokens:       {api_tokens['total_tokens']:>10,}")

print(f"\n2. Token Breakdown:")
print(f"   User Question:      {user_question_tokens:>10,} tokens")
print(f"   System Message:     {system_message_tokens:>10,} tokens")
print(f"   Context:            {context_tokens:>10,} tokens ({len(full_context):,} chars)")
print(f"   Response:           {response_tokens:>10,} tokens")

print(f"\n3. Test Data:")
print(f"   Query:              {test_query[:100]}...")
print(f"   Response preview:   {test_response[:100]}...")

print(f"\n4. Groundedness Evaluation:")
if groundedness_result:
    print(f"   Status:             SUCCESS ✓")
    print(f"   Score:              {groundedness_result.get('groundedness', 'N/A')}")
    if 'token_usage' in groundedness_result:
        print(f"   Actual tokens:      {groundedness_result['token_usage'].get('total_tokens', 'N/A'):,}")
else:
    print(f"   Status:             FAILED ❌")
    print(f"   Estimated tokens:   {eval_prompt_tokens:,}")

print(f"\n5. Configuration:")
print(f"   Base context:       {len(base_context):>10,} characters")
print(f"   Repetitions:        {CONTEXT_REPETITIONS:>10,}x")
print(f"   Model deployment:   {deployment}")

print("\n" + "=" * 80)

# Provide recommendations
print("\nRECOMMENDATIONS:")
print("-" * 80)

if context_tokens > 100000:
    print("⚠ Context is very large (>100K tokens)")
    print("  → Consider using GPT-4o or GPT-4-turbo (128K context window)")
    print("  → Or implement context chunking strategy")
elif context_tokens > 30000:
    print("⚠ Context is large (>30K tokens)")
    print("  → Ensure you're using a model with adequate context window")
    print("  → GPT-4-32k, GPT-4-turbo, or GPT-4o recommended")
elif context_tokens > 7000:
    print("⚠ Context approaching limits of smaller models")
    print("  → GPT-4 (8K) or larger recommended")
else:
    print("✓ Context size is manageable for most models")

print(f"\nTo test with larger contexts, increase CONTEXT_REPETITIONS")
print(f"Current: {CONTEXT_REPETITIONS}, Tokens: {context_tokens:,}")
print(f"If doubled ({CONTEXT_REPETITIONS * 2}): ~{context_tokens * 2:,} tokens")
print(f"If 5x ({CONTEXT_REPETITIONS * 5}): ~{context_tokens * 5:,} tokens")
print(f"If 10x ({CONTEXT_REPETITIONS * 10}): ~{context_tokens * 10:,} tokens")


## Testing Instructions

**To reproduce the 400 error:**

1. Gradually increase `CONTEXT_REPETITIONS` in the "Load and Prepare Context" cell
2. Rerun the cells from that point forward
3. Monitor the token counts and watch for when the evaluation fails
4. Compare against your model's context window limits

**Expected behavior:**
- With small contexts: Evaluation succeeds
- With contexts approaching model limits: May see slower performance
- With contexts exceeding model limits: Should see 400 (Bad Request) error

**Key metrics to watch:**
- `context_tokens`: The main driver of token consumption
- `estimated_eval_prompt_tokens`: Total estimated tokens for evaluation
- Model limit comparison: Check if tokens exceed ~90% of model's context window

**Suggested repetition values to test:**
- Start with: 1 (baseline)
- Moderate: 2-3x (test normal scenarios)
- Large: 5-10x (approach limits for GPT-4-32k)
- Very large: 20-50x (test GPT-4-turbo/GPT-4o limits or trigger errors on smaller models)

## References

- [Azure AI Evaluation Documentation](https://learn.microsoft.com/azure/ai-studio/how-to/evaluate-sdk)
- [Azure OpenAI Service Documentation](https://learn.microsoft.com/azure/ai-services/openai/)
- [Token Counting with tiktoken](https://github.com/openai/tiktoken)
- [Model Context Windows](https://platform.openai.com/docs/models)